# Question 1: Keyword location with MapReduce (R + Hadoop Streaming)

**Goal.** Find the **row and column** of every occurrence of a keyword in every document of the corpus `map_reduce_test_docs`. The search is case-insensitive and the keyword is supplied at run time.

**Analytics flow**
1. Design the job: choose the key/value pairs.
2. Map: emit `document <TAB> row:col` for each hit.
3. Shuffle and sort: group the pairs by document.
4. Reduce: one line per document with the hit count and sorted locations.
5. Run it with different keywords and match modes.
6. Verify against an independent search.

The mapper and reducer are the same `mapper.R` / `reducer.R` that `run_hadoop.sh` submits to a Hadoop cluster. This notebook runs them locally through `main.R`, which chains map, sort and reduce.

Full write-up, the layout diagram and the Hadoop configuration are in `src/q1_keyword_search_mapreduce/README.md`.

## 1. Design (Question 1.1)

**Key** = document name, **value** = `row:col`. The shuffle phase groups by key, so each reducer call receives everything found in one document.

![MapReduce layout](../src/q1_keyword_search_mapreduce/docs/mapreduce_layout.png)

## 2. Set up

In [ ]:
# Point main.R at the Q1 module and stop it from auto-running
q1_root <- normalizePath("../src/q1_keyword_search_mapreduce")
corpus  <- normalizePath("../data/q1_map_reduce_test_docs")

options(keyword_mr.root = q1_root, keyword_mr.no_autorun = TRUE)
source(file.path(q1_root, "main.R"))

list.files(corpus)

## 3. Mapper (Question 1.2)

Reads the keyword from an argument or the `KEYWORD` variable, escapes regex symbols, and emits one pair per match.

In [ ]:
cat(readLines(file.path(q1_root, "mapper.R")), sep = "\n")

## 4. Reducer (Question 1.3)

Sorts each document's locations numerically by (row, column), counts them and writes one line per document.

In [ ]:
cat(readLines(file.path(q1_root, "reducer.R")), sep = "\n")

## 5. Hadoop configuration (Question 1.4)

Settings such as one split per document (so row numbers start at line 1) and one reducer live in the XML file below. `run_hadoop.sh` submits the job.

In [ ]:
cat(readLines(file.path(q1_root, "config", "keyword-search-conf.xml")), sep = "\n")

## 6. Run the job: whole-word match for `africa`

In [ ]:
res <- run_job("africa", match_mode = "word", corpus = corpus, save = FALSE)

In [ ]:
res$summary

In [ ]:
head(res$locations, 10)

**Recorded output** (screenshot from the project documentation): keyword search run for 'africa'

![keyword search run for 'africa'](../docs/images/q1_run_output.png)

## 7. Substring mode also counts *African*

In [ ]:
res_sub <- run_job("africa", match_mode = "substring", corpus = corpus, verbose = FALSE, save = FALSE)
res_sub$summary

## 8. Nothing is hard-coded: another keyword, and a case check

In [ ]:
res_sars  <- run_job("sars",  corpus = corpus, verbose = FALSE, save = FALSE)
res_mixed <- run_job("AfRiCa", corpus = corpus, verbose = FALSE, save = FALSE)

head(res_sars$summary, 3)
cat("AfRiCa gives the same result as africa:", identical(res_mixed$summary, res$summary), "\n")

## 9. Hits per document

In [ ]:
barplot(setNames(res$summary$hits, res$summary$document),
        main = "Occurrences of 'africa' per document (whole-word mode)",
        ylab = "Hits", col = "steelblue")

## 10. Verification

`tests/verify.R` compares the job with an independent fixed-string search for several keywords, in both modes, on the folder, on the `.zip`, and on edge-case documents (accents, Windows line endings, a byte-order mark, blank lines).

In [ ]:
# Optional: runs the module's own test-suite
system2(file.path(R.home("bin"), "Rscript"), file.path(q1_root, "tests", "verify.R"))